# 01 — Collect data

Sources (see `config.SOURCES`):
- **`opensubtitles_enfi`** — EN-FI parallel pairs from OpenSubtitles (HF, streamed). Dialogue, leans colloquial.
- **`opus_100`** — EN-FI parallel pairs from OPUS-100 (HF, streamed). Broad, mixed-domain.
- **`genius_rap`** — Finnish rap lyrics, FI-only "flavor". Genius blocks lyric scraping (Cloudflare Private Access Tokens), so we use the Genius **API** for song lists only and you paste favourite lines into a **curated seed** (`data/raw/genius_rap/<artist>.txt`).

Config reads `.env` (see `.env.example`): `GENIUS_ACCESS_TOKEN`, `LMSTUDIO_*`.

In [ ]:
from puhekieli_llm.config import summary, SOURCES, RAP_ARTISTS
from puhekieli_llm import sources as S
print(summary())
print()
for name, m in SOURCES.items():
    print(f"{name:20s} role={m['role']:7s} register={m['register']:9s} {m['status']}")

## A) Rap lyrics — curated seed (flavor, FI-only)

Genius scraping is blocked, so this is a two-step manual flow:
1. Run the metadata cell to list song **URLs** per artist (uses the working Genius API).
2. Open URLs in your browser, paste lines you like into `data/raw/genius_rap/<artist>.txt` (one line per line; `#` lines tag songs). See `examples/genius_rap_seed.example.txt`.
3. Run the clean cell to build `data/clean/genius_rap.jsonl`.

In [ ]:
import os
# Step 1: list songs per artist via the Genius API (metadata only, no lyrics).
if os.getenv('GENIUS_ACCESS_TOKEN'):
    S.fetch_genius_metadata(RAP_ARTISTS, max_songs_per_artist=40)
else:
    print('No GENIUS_ACCESS_TOKEN in .env — skipping song listing.')
    print('You can still paste lyrics manually into data/raw/genius_rap/<artist>.txt')

In [ ]:
from puhekieli_llm.config import RAW, CLEAN
# Step 3: clean curated seed .txt files -> data/clean/genius_rap.jsonl
seed_dir = RAW / 'genius_rap'
txts = [f for f in seed_dir.glob('*.txt') if f.stem.lower() not in {'readme','example'}] if seed_dir.exists() else []
if txts:
    S.clean_genius_lyrics()
else:
    print('No seed lyrics yet. Paste lines into data/raw/genius_rap/<artist>.txt then re-run.')

In [ ]:
# Peek + puhekieli sanity check on what we collected
from puhekieli_llm.eval import puhekieli_score, corpus_puhekieli_rate
p = CLEAN / 'genius_rap.jsonl'
if p.exists():
    recs = list(S.read_jsonl(p))
    print(f'{len(recs)} lyric lines')
    for r in recs[:8]:
        print(f"  [{puhekieli_score(r['fi'])['score']:+.1f}] {r['fi']}")
    print(f"\nspoken-leaning fraction: {corpus_puhekieli_rate([r['fi'] for r in recs]):.0%}")
else:
    print('genius_rap.jsonl not built yet.')

## B) EN-FI parallel pairs (OpenSubtitles + OPUS-100)

Both stream from HuggingFace — no giant downloads, capped by `MAX_PAIRS`. Keep small while iterating; bump up for real training.

In [ ]:
from puhekieli_llm.config import RAW
MAX_PAIRS = 50_000
# OpenSubtitles EN-FI (dialogue). Streamed from HF; safe to run headless.
S.fetch_opensubtitles(max_pairs=MAX_PAIRS)

In [ ]:
q = CLEAN / 'opensubtitles_enfi.jsonl'
if q.exists():
    recs = list(S.read_jsonl(q))
    print(f'{len(recs)} EN-FI pairs')
    for r in recs[:6]:
        print(f"  EN: {r['en']}\n  FI: {r['fi']}\n")

In [ ]:
# OPUS-100 EN-FI (broad, mixed-domain) — complements the subtitle dialogue.
S.fetch_opus100(max_pairs=MAX_PAIRS)

o = CLEAN / 'opus_100.jsonl'
if o.exists():
    recs = list(S.read_jsonl(o))
    print(f'{len(recs)} EN-FI pairs')
    for r in recs[:4]:
        print(f"  EN: {r['en']}\n  FI: {r['fi']}\n")

## ✅ Next
- `01b_synthesize.ipynb` — back-translate rap lyrics into EN→FI pairs (local LLM).
- then `02_tokenizer.ipynb` — train a BPE tokenizer over subtitles + rap + synth.